# MARS Style Embeddings · Find & Group Photos by Visual Style (Version 2)

**How it works (MARS methodology):**
1. A Vision-Language Model looks at each photo and is asked: *"In one word, the style of this listing is: \""*
2. Instead of reading its text answer, we capture the model's **internal hidden state** at the open-quote token — the neural representation of style *before* it picks a word
3. This hidden vector (3584 numbers) is L2-normalized and stored in DuckDB as a style fingerprint
4. To find similar photos: compare fingerprints using *cosine similarity* (1.0 = identical style, 0.0 = nothing in common)
5. To group photos: K-means clustering sorts all fingerprints into buckets — photos in the same bucket share visual style

**Why this is different from V1:** V1 converts style to text first (`qwen3.5 → description → nomic-embed-text → 768-dim`). MARS skips the text step entirely — the hidden state *is* the style representation, making it more direct and potentially more faithful to visual nuance.

## Config & Imports

In [1]:
from dotenv import load_dotenv
load_dotenv(override=True)

# ── config ────────────────────────────────────────────────────
MODEL_ID      = "mlx-community/Qwen2.5-VL-7B-Instruct-4bit"  # already cached, no download needed
EMBED_DIM     = 3584          # hidden state dim for Qwen2.5-VL-7B
N_CLUSTERS    = 20
PROCESS_PHOTO = 20            # "All" | integer (top N) | "Kinan.Sweidan-1.jpg" (single file)
NUDGE_PROMPT  = 'In one word, the style of this listing is: "'

# ── imports ───────────────────────────────────────────────────
import hashlib, time
from pathlib import Path

import duckdb
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display as ipy_display, HTML
import mlx.core as mx
from mlx_vlm import load as mlx_load

RAW_DIR = Path("./data/raw")
_all = sorted(p for ext in ("*.jpg", "*.JPG") for p in RAW_DIR.glob(ext))
if PROCESS_PHOTO == "All":
    photos = _all
elif isinstance(PROCESS_PHOTO, int):
    photos = _all[:PROCESS_PHOTO]
else:
    photos = [RAW_DIR / PROCESS_PHOTO]
print(f"{len(photos)} photos selected")

SyntaxError: unterminated string literal (detected at line 9) (2297621444.py, line 9)

## Load Model

Run once per session. Uses the 4-bit MLX model already cached at `~/.cache/huggingface/hub/`. Loads in seconds on Apple Silicon.

In [ ]:
model, processor = mlx_load(MODEL_ID)
print(f"Loaded {MODEL_ID}")

Fetching 12 files: 100%|██████████| 12/12 [00:00<00:00, 199728.76it/s]


Loaded mlx-community/Qwen2.5-VL-7B-Instruct-4bit


## Database Setup

In [ ]:
con = duckdb.connect("./outputs/photos.duckdb")

con.execute(f"""
    CREATE TABLE IF NOT EXISTS mars_embeddings (
        photo_id   VARCHAR PRIMARY KEY,
        image_path VARCHAR NOT NULL,
        embedding  FLOAT[{EMBED_DIM}],
        created_at TIMESTAMP DEFAULT now()
    )
""")

count = con.execute("SELECT count(*) FROM mars_embeddings").fetchone()[0]
print(f"mars_embeddings table ready · {count} rows stored")

mars_embeddings table ready · 0 rows stored


## Helpers · Extract Style Embedding

In [ ]:
def photo_id(path: Path) -> str:
    """Stable ID: SHA-256 of first 64KB of file (first 16 hex chars)."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        h.update(f.read(65536))
    return h.hexdigest()[:16]


def extract_style_embedding(image_path: Path) -> np.ndarray:
    """MARS: capture the VLM hidden state at the nudge-token position.

    mlx_vlm's model.__call__ returns only logits (LanguageModelOutput.logits),
    so we bypass lm_head and call the transformer backbone directly:
      1. get_input_embeddings — merges visual tokens into text embeddings
      2. language_model.model — runs transformer, returns hidden states (batch, seq_len, hidden_dim)
    """
    image = Image.open(image_path).convert("RGB")

    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": NUDGE_PROMPT},
        ],
    }]

    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    inputs = processor(text=[text], images=[image], return_tensors="np")

    input_ids      = mx.array(inputs["input_ids"])
    pixel_values   = mx.array(inputs["pixel_values"]).astype(mx.float16)
    image_grid_thw = mx.array(inputs["image_grid_thw"])

    # Step 1: merge visual tokens into text embeddings + pre-compute position_ids
    features = model.get_input_embeddings(
        input_ids, pixel_values, image_grid_thw=image_grid_thw
    )

    # Step 2: run transformer backbone — returns (batch, seq_len, hidden_dim)
    # position_ids were stored on language_model by get_input_embeddings above
    h = model.language_model.model(
        input_ids,
        inputs_embeds=features.inputs_embeds,
        position_ids=model.language_model._position_ids,
    )
    mx.eval(h)

    # Last token position = the open-quote nudge point
    embedding = np.array(h[0, -1, :])

    # L2 normalization so cosine similarity = dot product
    embedding = embedding / np.linalg.norm(embedding)
    return embedding


def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b))


print("Helpers ready.")

Helpers ready.


## Ingest · Extract + Store All Embeddings

Run this once. Already-processed photos are skipped automatically. Expect ~2–5s per photo on MPS/CUDA.

In [ ]:
for path in photos:
    pid = photo_id(path)

    if con.execute("SELECT 1 FROM mars_embeddings WHERE photo_id = ?", [pid]).fetchone():
        print(f"  skip  {path.name}")
        continue

    t0 = time.time()
    emb = extract_style_embedding(path)
    elapsed = time.time() - t0

    con.execute(
        "INSERT INTO mars_embeddings (photo_id, image_path, embedding) VALUES (?, ?, ?)",
        [pid, str(path.resolve()), emb.tolist()],
    )
    print(f"  ✓  {path.name}  {elapsed:.1f}s  dim={emb.shape[0]}  norm={np.linalg.norm(emb):.4f}")

total = con.execute("SELECT count(*) FROM mars_embeddings").fetchone()[0]
print(f"\nDone · {total} photos in mars_embeddings table")

  ✓  Kinan.Sweidan-1.jpg  11.5s  dim=3584  norm=0.0000

Done · 1 photos in mars_embeddings table


/Users/kinan/Dev/photo-pipeline/.venv/lib/python3.11/site-packages/numpy/linalg/_linalg.py:2767: RuntimeWarning: overflow encountered in dot
  sqnorm = x.dot(x)


## Find Similar Photos

Change `QUERY_PHOTO` to any filename in `data/raw/` and run this cell.

In [ ]:
QUERY_PHOTO = "Kinan.Sweidan-12.jpg"   # ← change this to any photo
TOP_N       = 4                         # how many similar photos to return

query_path = RAW_DIR / QUERY_PHOTO
query_pid  = photo_id(query_path)

row = con.execute(
    "SELECT embedding FROM mars_embeddings WHERE photo_id = ?",
    [query_pid],
).fetchone()

if row is None:
    print(f"'{QUERY_PHOTO}' not yet ingested — run the Ingest cell first.")
else:
    q_emb = np.array(row[0], dtype=np.float32)

    rows = con.execute(
        "SELECT photo_id, image_path, embedding FROM mars_embeddings WHERE photo_id != ?",
        [query_pid],
    ).fetchall()

    scored = [
        (cosine_sim(q_emb, np.array(r[2], dtype=np.float32)), r[1])
        for r in rows
    ]
    scored.sort(reverse=True)
    top = scored[:TOP_N]

    # ── Display ────────────────────────────────────────────────
    ncols = TOP_N + 1
    fig, axes = plt.subplots(1, ncols, figsize=(3.5 * ncols, 5))

    axes[0].imshow(Image.open(query_path), cmap="gray")
    axes[0].set_title(f"QUERY\n{QUERY_PHOTO}", fontsize=8, color="#333", fontweight="bold")
    axes[0].axis("off")

    for i, (score, img_path) in enumerate(top, start=1):
        name = Path(img_path).name
        axes[i].imshow(Image.open(img_path), cmap="gray")
        axes[i].set_title(f"#{i}  similarity: {score:.3f}\n{name}", fontsize=8, color="#555")
        axes[i].axis("off")

    plt.suptitle(f"MARS · Photos most similar to '{QUERY_PHOTO}'", fontsize=11, y=1.01)
    plt.tight_layout()
    plt.show()

'Kinan.Sweidan-12.jpg' not yet ingested — run the Ingest cell first.


## Visual Grouping · Cluster Photos by Style

K-means groups all photos into `N_CLUSTERS` buckets based on their MARS style embeddings. Try changing `N_CLUSTERS` (top of notebook) and re-running.

In [ ]:
from sklearn.cluster import KMeans

rows = con.execute(
    "SELECT photo_id, image_path, embedding FROM mars_embeddings ORDER BY photo_id"
).fetchall()

if len(rows) < N_CLUSTERS:
    print(f"Need at least {N_CLUSTERS} photos ingested. Run Ingest cell first.")
else:
    paths  = [r[1] for r in rows]
    # Embeddings are already L2-normalized, so K-means on them respects cosine similarity
    matrix = np.array([r[2] for r in rows], dtype=np.float32)

    km = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init="auto")
    labels = km.fit_predict(matrix)

    clusters = {i: [] for i in range(N_CLUSTERS)}
    for label, path in zip(labels, paths):
        clusters[label].append(path)

    for cluster_id, cluster_paths in sorted(clusters.items()):
        n = len(cluster_paths)
        fig, axes = plt.subplots(1, n, figsize=(3 * n, 4))
        if n == 1:
            axes = [axes]

        for ax, img_path in zip(axes, cluster_paths):
            ax.imshow(Image.open(img_path), cmap="gray")
            ax.set_title(Path(img_path).name, fontsize=7, color="#555")
            ax.axis("off")

        fig.suptitle(
            f"Group {cluster_id + 1} · {n} photo{'s' if n != 1 else ''}",
            fontsize=11, fontweight="bold", y=1.02,
        )
        plt.tight_layout()
        plt.show()

Need at least 20 photos ingested. Run Ingest cell first.


## Export · HTML

In [ ]:
import subprocess, sys, datetime

out_dir = Path("./outputs/html")
out_dir.mkdir(parents=True, exist_ok=True)

out_file = out_dir / f"similarity_search_v2_{datetime.date.today()}.html"

result = subprocess.run(
    [sys.executable, "-m", "nbconvert", "--to", "html", "--no-input",
     "similarity_search_v2.ipynb", "--output", str(out_file.resolve())],
    capture_output=True, text=True,
)
if result.returncode == 0:
    print(f"✓ Exported to {out_file}")
else:
    print(result.stderr)

✓ Exported to outputs/html/similarity_search_v2_2026-04-21.html
